In [1]:
import json
import os
import pathlib
from pathlib import Path

import dask.array as da
import napari
import natsort
import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import tifffile
from napari.utils.notebook_display import nbscreenshot
from rich.pretty import pprint
from tifffile import imread
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from ultrack import (
    MainConfig,
    add_flow,
    link,
    segment,
    solve,
    to_ctc,
    to_tracks_layer,
    tracks_to_zarr,
)
from ultrack.imgproc import detect_foreground, robust_invert
from ultrack.imgproc.flow import (
    advenct_from_quasi_random,
    timelapse_flow,
    trajectories_to_tracks,
)
from ultrack.tracks.stats import tracks_df_movement
from ultrack.utils.array import array_apply, create_zarr
from ultrack.utils.cuda import on_gpu

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

In [2]:
# def clear_gpu_memory():
#     """
#     A little function to clear gpu memory
#     """

#     torch.cuda.empty_cache()


# clear_gpu_memory()

# begin time, CPU memory peak usage and GPU memory peak usage tracking
import time
import tracemalloc

# import pynvml

# pynvml.nvmlInit()

start_time = time.time()
tracemalloc.start()
# handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # Assuming you want to track GPU 0

In [3]:
if not in_notebook:
    print("Running as script")
    # set up arg parser
    parser = argparse.ArgumentParser(description="Segment the nuclei of a tiff image")

    parser.add_argument(
        "--well_fov",
        type=str,
        help="Path to the input directory containing the tiff images",
    )
    parser.add_argument(
        "--generate_gif",
        action="store_true",
        help="Whether to generate a gif of the tracks",
    )

    parser.add_argument(
        "--plate_name",
        type=str,
        help="Name of the plate to process (e.g., 'Wave2')",
    )

    args = parser.parse_args()
    well_fov = args.well_fov
    generate_gif = args.generate_gif
    plate_name = args.plate_name
    if generate_gif:
        print("GIF generation is enabled, this may take a while...")

else:
    print("Running in a notebook")
    well_fov = "B2_1"  # example well_fov
    generate_gif = True
    plate_name = "plate_2"  # example plate name

image_base_dir = bandicoot_check(
    root_dir=root_dir,
    bandicoot_mount_path=pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(),
)

raw_image_input_dir = pathlib.Path(
    image_base_dir / "processed_data" / "0.renamed_files" / plate_name / well_fov
).resolve(strict=True)

segmentation_mask_input_dir = pathlib.Path(
    image_base_dir
    / "processed_data"
    / "2.cell_segmentation_masks"
    / plate_name
    / well_fov
).resolve()

config_output_path = "../results/ultrack_config.json"
# with open(config_output_path, "r") as f:
#     config_dict = json.load(f)
# config = MainConfig.parse_obj(config_dict)

temporary_output_dir = pathlib.Path("../tmp_output").resolve()
figures_output_dir = pathlib.Path("../figures").resolve()
results_output_dir = pathlib.Path("../results").resolve()
temporary_output_dir.mkdir(exist_ok=True)
figures_output_dir.mkdir(exist_ok=True)
results_output_dir.mkdir(exist_ok=True)
config = MainConfig()
pprint(config)

Running in a notebook


MainConfig(
│   data_config=DataConfig(
│   │   n_workers=1,
│   │   working_dir=PosixPath('.'),
│   │   database_file_name='data.db',
│   │   database='sqlite',
│   │   address=None,
│   │   in_memory_db_id=0
│   ),
│   segmentation_config=SegmentationConfig(
│   │   min_area=100,
│   │   min_area_factor=4.0,
│   │   max_area=1000000,
│   │   n_workers=1,
│   │   min_frontier=0.0,
│   │   threshold=0.5,
│   │   max_noise=0.0,
│   │   random_seed='frame',
│   │   ws_hierarchy=<function watershed_hierarchy_by_area at 0x7233af738b80>,
│   │   anisotropy_penalization=0.0
│   ),
│   linking_config=LinkingConfig(
│   │   max_distance=15.0,
│   │   n_workers=1,
│   │   max_neighbors=5,
│   │   distance_weight=0.0,
│   │   z_score_threshold=5.0
│   ),
│   tracking_config=TrackingConfig(
│   │   solver_name='',
│   │   appear_weight=-0.001,
│   │   disappear_weight=-0.001,
│   │   division_weight=-0.001,
│   │   image_border_size=None,
│   │   n_threads=-1,
│   │   window_size=None,
│   │   overlap_size=1,
│   │   solution_gap=0.001,
│   │   time_limit=36000,
│   │   method=0,
│   │   link_function='power',
│   │   power=4,
│   │   bias=-0.0,
│   │   dismiss_weight_guess=None,
│   │   include_weight_guess=None
│   )
)

In [4]:
file_extensions = {".tif", ".tiff"}
# get all the raw image files
raw_images = list(raw_image_input_dir.glob("*"))
raw_images = [f for f in raw_images if f.suffix in file_extensions]
raw_images = sorted(raw_images)

# get all the segmentation mask files
segmentation_masks = list(segmentation_mask_input_dir.glob("*"))
segmentation_masks = [f for f in segmentation_masks if f.suffix in file_extensions]
segmentation_masks = sorted(segmentation_masks)


nuclei_files = [f for f in raw_images if "C4" in f.name.split("_")[3]]
mask_files = [f for f in segmentation_masks if "nuclei" in f.name]

nuclei_files = natsort.natsorted(nuclei_files)
mask_files = natsort.natsorted(mask_files)

print(f"Found {len(mask_files)} segmentation mask files in the input directory")
print(f"Found {len(nuclei_files)} nuclei files in the input directory")

Found 288 segmentation mask files in the input directory
Found 288 nuclei files in the input directory


In [5]:
# read in the masks and create labels
masks = []
for tiff_file in mask_files[:50]:
    img = tifffile.imread(tiff_file)
    masks.append(img)

masks = np.array(masks)

nuclei = []
for tiff_file in nuclei_files[:50]:
    img = tifffile.imread(tiff_file)
    nuclei.append(img)
nuclei = np.array(nuclei)

In [6]:
image_dims = nuclei[0].shape

In [7]:
image = masks

viewer = napari.Viewer()

im_layer = viewer.add_image(nuclei)
image = viewer.layers[0].data

In [8]:
# detection = create_zarr(image.shape, bool, "detection.zarr", overwrite=True)
# array_apply(
#     image,
#     out_array=detection,
#     func=on_gpu(detect_foreground),
# )

# viewer.add_labels(detection, visible=True).contour = 2
detection = masks
viewer.add_labels(detection, visible=True)

<Labels layer 'detection' at 0x7232c51b6e10>

In [9]:
boundaries = create_zarr(image.shape, np.float16, "boundaries.zarr", overwrite=True)
array_apply(
    image,
    out_array=boundaries,
    func=on_gpu(robust_invert),
    sigma=3.0,
)

viewer.add_image(boundaries, visible=False)

Applying robust_invert ...: 100%|██████████| 50/50 [00:06<00:00,  8.32it/s]


<Image layer 'boundaries' at 0x7232be1b2210>

In [10]:
if pathlib.Path("flow.zarr").exists():
    !rm -r flow.zarr # removing previous flow
flow = timelapse_flow(
    image, store_or_path="flow.zarr", n_scales=2, lr=1e-2, num_iterations=2_000
)
viewer.add_image(
    flow,
    contrast_limits=(-0.001, 0.001),
    colormap="turbo",
    visible=False,
    scale=(4,) * 3,
    channel_axis=1,
    name="flow field",
)

Computing flow: 100%|██████████| 49/49 [09:42<00:00, 11.89s/it]


[<Image layer 'flow field' at 0x7232701e31a0>,
 <Image layer 'flow field [1]' at 0x7232be1a6b40>]

In [11]:
trajectory = advenct_from_quasi_random(flow, detection.shape[-2:], n_samples=1000)
flow_tracklets = pd.DataFrame(
    trajectories_to_tracks(trajectory),
    columns=["track_id", "t", "y", "x"],
)
flow_tracklets[
    ["y", "x"]
] += 0.5  # napari was crashing otherwise, might be an openGL issue
flow_tracklets[["dy", "dx"]] = tracks_df_movement(flow_tracklets)
flow_tracklets["angles"] = np.arctan2(flow_tracklets["dy"], flow_tracklets["dx"])

flow_tracklets.to_csv("flow_tracklets.csv", index=False)

viewer.add_tracks(
    flow_tracklets[["track_id", "t", "y", "x"]],
    name="flow vectors",
    visible=True,
    tail_length=25,
    features=flow_tracklets[["angles", "dy", "dx"]],
    colormap="hsv",
).color_by = "angles"

# nbscreenshot(viewer)

In [12]:
cfg = MainConfig()

cfg.data_config.n_workers = 1

cfg.segmentation_config.n_workers = 1
cfg.segmentation_config.min_area = 250
cfg.segmentation_config.max_area = 15_000

cfg.linking_config.n_workers = 1
cfg.linking_config.max_neighbors = 5
cfg.linking_config.max_distance = 20.0

cfg.tracking_config.window_size = 70
cfg.tracking_config.overlap_size = 5
cfg.tracking_config.appear_weight = -0.01
cfg.tracking_config.disappear_weight = -0.001
cfg.tracking_config.division_weight = 0

pprint(cfg)

MainConfig(
│   data_config=DataConfig(
│   │   n_workers=1,
│   │   working_dir=PosixPath('.'),
│   │   database_file_name='data.db',
│   │   database='sqlite',
│   │   address=None,
│   │   in_memory_db_id=0
│   ),
│   segmentation_config=SegmentationConfig(
│   │   min_area=250,
│   │   min_area_factor=4.0,
│   │   max_area=15000,
│   │   n_workers=1,
│   │   min_frontier=0.0,
│   │   threshold=0.5,
│   │   max_noise=0.0,
│   │   random_seed='frame',
│   │   ws_hierarchy=<function watershed_hierarchy_by_area at 0x7233af738b80>,
│   │   anisotropy_penalization=0.0
│   ),
│   linking_config=LinkingConfig(
│   │   max_distance=20.0,
│   │   n_workers=1,
│   │   max_neighbors=5,
│   │   distance_weight=0.0,
│   │   z_score_threshold=5.0
│   ),
│   tracking_config=TrackingConfig(
│   │   solver_name='',
│   │   appear_weight=-0.01,
│   │   disappear_weight=-0.001,
│   │   division_weight=0,
│   │   image_border_size=None,
│   │   n_threads=-1,
│   │   window_size=70,
│   │   overlap_size=5,
│   │   solution_gap=0.001,
│   │   time_limit=36000,
│   │   method=0,
│   │   link_function='power',
│   │   power=4,
│   │   bias=-0.0,
│   │   dismiss_weight_guess=None,
│   │   include_weight_guess=None
│   )
)

In [13]:
segment(detection, boundaries, cfg, overwrite=True)
add_flow(cfg, flow)

Adding nodes to database:   0%|          | 0/50 [00:00<?, ?it/s]/home/lippincm/miniforge3/envs/timelapse_tracking_env/lib/python3.12/site-packages/ultrack/core/segmentation/hierarchy.py:69: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  morphology.remove_small_objects(
/home/lippincm/miniforge3/envs/timelapse_tracking_env/lib/python3.12/site-packages/ultrack/core/segmentation/vendored/hierarchy.py:356: FutureWarning: `RegionProperties.intensity_image` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.image_intensity` instead. 
  graph, weights = mask_to_graph(c.image, c.intensity_image, anisotropy_pen)
/

In [14]:
import pandas as pd

pd.options.mode.copy_on_write = False
import numpy as np
import skimage.util._map_array as map_array_mod

_original_map_array = map_array_mod.map_array


def _patched_map_array(input_arr, input_vals, output_vals, out=None):
    input_arr = np.array(input_arr, copy=True)
    input_vals = np.array(input_vals, copy=True)
    output_vals = np.array(output_vals, copy=True)
    return _original_map_array(input_arr, input_vals, output_vals, out=out)


map_array_mod.map_array = _patched_map_array

/tmp/ipykernel_1589181/1362668334.py:2: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  pd.options.mode.copy_on_write = False


In [15]:
link(cfg, overwrite=True)
solve(cfg)

Linking nodes.: 100%|██████████| 49/49 [01:20<00:00,  1.65s/it]


Using Coin-OR Branch and Cut solver
Solving ILP batch 0
Constructing ILP ...
Solving ILP ...
Welcome to the CBC MILP Solver 
Version: devel 
Build Date: Mar 23 2026
Starting solution of the Linear programming relaxation problem using Primal Simplex

Coin0506I Presolve 453847 (-32557) rows, 642017 (-27968) columns and 1527555 (-70306) elements
Clp0030I 1 infeas 56382.392, obj 8276.7815 - mu 9.688901, its 2, 285156 interior
Clp0030I 2 infeas 30850.179, obj 7896.0594 - mu 4.7475615, its 52, 329764 interior
Clp0030I 3 infeas 16709.582, obj 7445.9405 - mu 1.5823623, its 52, 376733 interior
Clp0030I 4 infeas 4388.4202, obj 5780.5247 - mu 1.5823623, its 52, 388023 interior
Clp0030I 5 infeas 1784.6406, obj 5223.3858 - mu 1.5823623, its 52, 391235 interior
Clp0030I 6 infeas 6310.1942, obj 6113.3972 - mu 0.52740134, its 52, 405164 interior
Clp0030I 7 infeas 922.32242, obj 5106.4876 - mu 0.52740134, its 62, 412596 interior
Clp0030I 8 infeas 340.85657, obj 5028.0624 - mu 0.52740134, its 52, 413697

In [16]:
tracks_df, graph = to_tracks_layer(cfg)
tracks_df.to_csv("tracks.csv", index=False)

segments = tracks_to_zarr(
    cfg,
    tracks_df,
    store_or_path="segments.zarr",
    overwrite=True,
)

viewer.layers["flow vectors"].visible = False
# viewer.layers["detection"].visible = False
viewer.add_tracks(
    tracks_df[["track_id", "t", "y", "x"]],
    name="tracks",
    graph=graph,
    visible=True,
)

viewer.add_labels(da.from_zarr(segments), name="segments").contour = 2

# nbscreenshot(viewer)

Exporting segmentation masks: 100%|██████████| 50/50 [00:12<00:00,  3.95it/s]


In [17]:
tracks_df["track_id"].unique()

array([   1,    2,    3, ..., 6018, 6019, 6020])

In [18]:
np.unique(masks)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18